In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

#block 1

In [ ]:
spark = SparkSession.builder.master("local[*]").appName("exercise_spark").getOrCreate()

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .option("delimiter", ";")
    .csv("../data/01.csv")
    )

In [ ]:
df.printSchema()

In [ ]:
df.show(4)

#block 2

In [ ]:
df.columns
df.where(F.col("Valor de Compra").isNotNull()).count()

In [ ]:
df.withColumn("Valor de Venda Copia", F.col("Valor de Venda")).show()

In [ ]:
df_minimized = df.select("Estado - Sigla", "Produto", "Valor de Venda", "Unidade de Medida")

In [ ]:
df_minimized = df_minimized.withColumnRenamed(
    "Valor de Venda", 
    "valor_de_venda"
)

df_minimized = df_minimized.withColumn("valor_de_venda", F.regexp_replace(F.col("valor_de_venda"), ",",".").cast("float"))


df_minimized.schema["valor_de_venda"].dataType

#block 3

In [ ]:
df_rename = df_minimized.toDF(*[cname.lower().strip().replace("  ", "_").replace(" ", "_") for cname in df_minimized.columns])

In [ ]:
df_rename.createOrReplaceTempView("values_fuel")

In [ ]:
spark.sql("""select * from values_fuel where valor_de_compra is not null""")